# Painting with Data

One of the most iconic data visualizations of the past decade is the [climate spiral](https://en.wikipedia.org/wiki/Climate_spiral), first published by climate scientist Ed Hawkins in 2016. It maps monthly global temperature anomalies onto a polar coordinate system, where each year traces a ring and the spiral expands outward as the planet warms.

In this notebook we recreate that idea using the p5.js kernel, importing the [chroma-js](https://gka.github.io/chroma.js/) color library from npm to handle perceptually uniform color mapping. The result is a data-driven generative piece: part chart, part art.

In [ ]:
import chroma from 'chroma-js';

## The Data

We use monthly global surface temperature anomalies from [NASA GISTEMP v4](https://data.giss.nasa.gov/gistemp/). The raw data was downloaded from NASA's [table data page](https://data.giss.nasa.gov/gistemp/tabledata_v4/GLB.Ts+dSST.csv) and embedded directly in the notebook to avoid CORS restrictions that prevent fetching it at runtime. Each value is the deviation in °C from the 1951–1980 baseline. Starting from 1980 — the tail end of that baseline period — the record captures the full arc of modern warming, from near-zero anomalies through the sharp acceleration that pushes well past +1.0 °C by the 2020s.

In [ ]:
const startYear = 1980;
const monthNames = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'];

// NASA GISTEMP v4 monthly anomalies (°C vs 1951-1980 baseline)
// source: https://data.giss.nasa.gov/gistemp/tabledata_v4/GLB.Ts+dSST.csv
// 552 values, 46 years (1980-2025), 12 months per row
const raw = [
   0.29, 0.39, 0.29, 0.30, 0.35, 0.20, 0.22, 0.18, 0.20, 0.13, 0.29, 0.21, // 1980
   0.53, 0.42, 0.48, 0.32, 0.24, 0.29, 0.32, 0.35, 0.14, 0.12, 0.23, 0.41, // 1981
   0.05, 0.15, 0.03, 0.15, 0.18, 0.05, 0.14, 0.03, 0.14, 0.13, 0.18, 0.42, // 1982
   0.53, 0.43, 0.42, 0.28, 0.34, 0.22, 0.18, 0.35, 0.37, 0.17, 0.30, 0.16, // 1983
   0.31, 0.14, 0.26, 0.06, 0.33, 0.02, 0.19, 0.19, 0.21, 0.13, 0.06,-0.04, // 1984
   0.22,-0.04, 0.17, 0.12, 0.14, 0.15, 0.04, 0.17, 0.13, 0.12, 0.05, 0.14, // 1985
   0.26, 0.37, 0.30, 0.22, 0.21, 0.12, 0.11, 0.16, 0.03, 0.15, 0.10, 0.13, // 1986
   0.32, 0.43, 0.18, 0.24, 0.25, 0.35, 0.40, 0.25, 0.35, 0.33, 0.29, 0.46, // 1987
   0.57, 0.44, 0.52, 0.43, 0.44, 0.40, 0.33, 0.39, 0.37, 0.38, 0.12, 0.29, // 1988
   0.12, 0.30, 0.36, 0.29, 0.18, 0.17, 0.35, 0.33, 0.34, 0.29, 0.20, 0.37, // 1989
   0.42, 0.44, 0.80, 0.56, 0.46, 0.37, 0.46, 0.34, 0.23, 0.45, 0.46, 0.41, // 1990
   0.43, 0.50, 0.36, 0.51, 0.34, 0.53, 0.48, 0.39, 0.44, 0.29, 0.30, 0.32, // 1991
   0.48, 0.40, 0.48, 0.28, 0.31, 0.26, 0.09, 0.08,-0.01, 0.07, 0.03, 0.22, // 1992
   0.35, 0.37, 0.36, 0.28, 0.28, 0.23, 0.25, 0.11, 0.12, 0.23, 0.04, 0.18, // 1993
   0.26, 0.02, 0.29, 0.41, 0.28, 0.43, 0.30, 0.21, 0.31, 0.43, 0.44, 0.39, // 1994
   0.52, 0.79, 0.46, 0.47, 0.27, 0.43, 0.45, 0.45, 0.33, 0.46, 0.45, 0.26, // 1995
   0.24, 0.46, 0.33, 0.33, 0.27, 0.25, 0.37, 0.48, 0.25, 0.22, 0.42, 0.37, // 1996
   0.32, 0.41, 0.52, 0.34, 0.35, 0.55, 0.34, 0.41, 0.52, 0.61, 0.65, 0.59, // 1997
   0.58, 0.88, 0.63, 0.64, 0.68, 0.77, 0.66, 0.65, 0.42, 0.42, 0.43, 0.55, // 1998
   0.48, 0.64, 0.32, 0.32, 0.26, 0.36, 0.38, 0.31, 0.39, 0.34, 0.37, 0.41, // 1999
   0.25, 0.56, 0.55, 0.56, 0.36, 0.40, 0.39, 0.42, 0.39, 0.27, 0.30, 0.28, // 2000
   0.45, 0.44, 0.55, 0.50, 0.57, 0.52, 0.59, 0.49, 0.51, 0.50, 0.72, 0.56, // 2001
   0.77, 0.78, 0.88, 0.58, 0.64, 0.53, 0.62, 0.54, 0.64, 0.54, 0.59, 0.44, // 2002
   0.75, 0.58, 0.60, 0.55, 0.61, 0.48, 0.58, 0.65, 0.62, 0.72, 0.53, 0.75, // 2003
   0.58, 0.73, 0.63, 0.61, 0.37, 0.44, 0.26, 0.45, 0.50, 0.61, 0.73, 0.51, // 2004
   0.75, 0.60, 0.74, 0.67, 0.63, 0.65, 0.61, 0.60, 0.72, 0.76, 0.74, 0.68, // 2005
   0.56, 0.74, 0.63, 0.47, 0.48, 0.66, 0.54, 0.71, 0.66, 0.71, 0.74, 0.79, // 2006
   1.02, 0.70, 0.72, 0.77, 0.69, 0.61, 0.60, 0.60, 0.61, 0.59, 0.59, 0.50, // 2007
   0.30, 0.38, 0.75, 0.54, 0.50, 0.49, 0.60, 0.46, 0.61, 0.67, 0.70, 0.54, // 2008
   0.65, 0.53, 0.53, 0.61, 0.66, 0.64, 0.74, 0.69, 0.72, 0.65, 0.80, 0.67, // 2009
   0.75, 0.84, 0.92, 0.85, 0.76, 0.67, 0.63, 0.67, 0.63, 0.71, 0.81, 0.45, // 2010
   0.53, 0.48, 0.66, 0.64, 0.53, 0.63, 0.70, 0.75, 0.56, 0.65, 0.59, 0.60, // 2011
   0.47, 0.48, 0.57, 0.73, 0.78, 0.65, 0.59, 0.66, 0.72, 0.80, 0.78, 0.53, // 2012
   0.71, 0.62, 0.67, 0.53, 0.60, 0.70, 0.60, 0.71, 0.77, 0.69, 0.83, 0.70, // 2013
   0.77, 0.55, 0.78, 0.81, 0.87, 0.67, 0.58, 0.84, 0.88, 0.80, 0.67, 0.79, // 2014
   0.87, 0.91, 0.96, 0.75, 0.80, 0.81, 0.73, 0.80, 0.84, 1.08, 1.06, 1.17, // 2015
   1.18, 1.37, 1.35, 1.09, 0.96, 0.81, 0.85, 1.02, 0.90, 0.87, 0.91, 0.86, // 2016
   1.02, 1.14, 1.17, 0.94, 0.91, 0.71, 0.82, 0.88, 0.74, 0.89, 0.87, 0.93, // 2017
   0.82, 0.85, 0.88, 0.89, 0.81, 0.77, 0.84, 0.78, 0.80, 1.02, 0.84, 0.93, // 2018
   0.93, 0.96, 1.17, 1.01, 0.86, 0.91, 0.94, 0.95, 0.93, 1.00, 0.99, 1.12, // 2019
   1.18, 1.24, 1.18, 1.12, 1.00, 0.91, 0.89, 0.86, 0.97, 0.87, 1.08, 0.80, // 2020
   0.82, 0.64, 0.89, 0.76, 0.79, 0.85, 0.92, 0.81, 0.93, 0.99, 0.93, 0.88, // 2021
   0.91, 0.89, 1.05, 0.84, 0.84, 0.92, 0.94, 0.95, 0.89, 0.97, 0.73, 0.80, // 2022
   0.88, 0.97, 1.23, 0.99, 0.94, 1.09, 1.20, 1.19, 1.48, 1.34, 1.41, 1.37, // 2023
   1.25, 1.45, 1.39, 1.31, 1.15, 1.21, 1.20, 1.29, 1.24, 1.33, 1.30, 1.26, // 2024
   1.38, 1.26, 1.37, 1.25, 1.08, 1.06, 1.02, 1.18, 1.25, 1.20, 1.21, 1.06, // 2025
];

const data = raw.map((anomaly, i) => ({
  year: startYear + Math.floor(i / 12),
  month: i % 12,
  anomaly,
  index: i,
}));

const endYear = startYear + Math.floor((raw.length - 1) / 12);
const totalYears = endYear - startYear + 1;

// Render as HTML table
const css = 'border-collapse:collapse;font-family:monospace;font-size:13px;width:100%';
const thStyle = 'padding:6px 10px;text-align:right;border-bottom:2px solid #444;color:#8aa';
const thLeft = 'padding:6px 10px;text-align:left;border-bottom:2px solid #444;color:#8aa';
const tdStyle = 'padding:4px 10px;border-bottom:1px solid #333;text-align:right';
const tdLeft = 'padding:4px 10px;border-bottom:1px solid #333;text-align:left;font-weight:bold';

function anomColor(v) {
  if (v >= 1.2) return '#b2182b';
  if (v >= 0.9) return '#ef8a62';
  if (v >= 0.6) return '#fddbc7';
  if (v >= 0.3) return '#d1e5f0';
  if (v >= 0.0) return '#67a9cf';
  return '#2166ac';
}

const header = '<tr>' +
  '<th style="' + thLeft + '">Year</th>' +
  monthNames.map(m => '<th style="' + thStyle + '">' + m + '</th>').join('') +
  '<th style="' + thStyle + ';font-weight:bold">Avg</th>' +
  '</tr>';

const rows = [];
for (let y = 0; y < totalYears; y++) {
  const yr = startYear + y;
  const vals = raw.slice(y * 12, y * 12 + 12);
  const avg = +(vals.reduce((a, b) => a + b, 0) / vals.length).toFixed(2);
  rows.push(
    '<tr>' +
    '<td style="' + tdLeft + '">' + yr + '</td>' +
    vals.map(v =>
      '<td style="' + tdStyle + ';color:' + anomColor(v) + '">' + v.toFixed(2) + '</td>'
    ).join('') +
    '<td style="' + tdStyle + ';font-weight:bold;color:' + anomColor(avg) + '">' + avg.toFixed(2) + '</td>' +
    '</tr>'
  );
}

const globalMin = Math.min(...raw).toFixed(2);
const globalMax = Math.max(...raw).toFixed(2);
const globalMean = (raw.reduce((a, b) => a + b, 0) / raw.length).toFixed(2);

'<table style="' + css + '">' +
  '<thead>' + header + '</thead>' +
  '<tbody>' + rows.join('') + '</tbody>' +
  '<tfoot><tr>' +
    '<td style="' + tdLeft + '">' + totalYears + ' years</td>' +
    '<td style="' + tdStyle + '" colspan="12">' + raw.length + ' months \u00b7 min ' + globalMin + '\u00b0C \u00b7 max ' + globalMax + '\u00b0C</td>' +
    '<td style="' + tdStyle + ';font-weight:bold;color:' + anomColor(+globalMean) + '">' + globalMean + '</td>' +
  '</tr></tfoot>' +
'</table>'

The dataset begins at 1980, right at the close of the 1951–1980 baseline period. The earliest years hover near zero — close to the baseline average. A steady rise through the 1990s and 2000s accelerates sharply in the 2010s, and by 2023–2025 anomalies routinely exceed +1.2 °C — a level that seemed distant just decades ago.

## Color Science with chroma-js

Mapping temperature to color sounds simple, but naive RGB interpolation produces muddy intermediate tones. The [chroma-js](https://gka.github.io/chroma.js/) library — imported directly from npm — gives us perceptually uniform interpolation in the CIELAB color space. A scale from deep blue through white to warm red mirrors the intuitive association between temperature and color, with no perceptual dead zones in between.

In [ ]:
const minAnom = -0.60;
const maxAnom = 1.60;

const tempScale = chroma
  .scale(['#08306b', '#2166ac', '#67a9cf', '#d1e5f0', '#fddbc7', '#ef8a62', '#b2182b', '#67001f'])
  .domain([-0.5, -0.2, 0.0, 0.3, 0.6, 0.9, 1.2, 1.5])
  .mode('lab');

const thresholds = [0, 0.5, 1.0, 1.5];

const params = {
  lineWeight: 2,
  glowLayers: 3,
  glowSpread: 4,
  dotSize: 7,
  bgColor: [8, 12, 24],
  labelColor: [180, 190, 210],
};

The scale is anchored at eight key temperatures, spanning the full range from the coolest early decades through the warmest recent years. Between these stops, chroma-js interpolates through CIELAB space — producing smooth transitions that our eyes perceive as even gradients:

In [ ]:
const stops = [
  { anomaly: -0.5, hex: '#08306b', note: 'coldest in early record' },
  { anomaly: -0.2, hex: '#2166ac' },
  { anomaly:  0.0, hex: '#67a9cf', note: '1951\u20131980 baseline' },
  { anomaly:  0.3, hex: '#d1e5f0' },
  { anomaly:  0.6, hex: '#fddbc7' },
  { anomaly:  0.9, hex: '#ef8a62' },
  { anomaly:  1.2, hex: '#b2182b', note: 'crossing +1\u00B0C' },
  { anomaly:  1.5, hex: '#67001f', note: 'warmest in dataset' },
];

const scaleCss = 'border-collapse:collapse;font-family:monospace;font-size:13px';
const scaleTh = 'padding:6px 10px;text-align:left;border-bottom:2px solid #444;color:#8aa';
const scaleTd = 'padding:5px 10px;border-bottom:1px solid #333';

const scaleHeader = '<tr>' +
  ['', 'Anomaly', 'Hex', 'Note'].map(h => '<th style="' + scaleTh + '">' + h + '</th>').join('') +
  '</tr>';

const scaleRows = stops.map(s =>
  '<tr>' +
  '<td style="' + scaleTd + '">' +
    '<div style="width:36px;height:18px;border-radius:3px;background:' + s.hex + '"></div>' +
  '</td>' +
  '<td style="' + scaleTd + ';color:' + s.hex + ';font-weight:bold">' + (s.anomaly >= 0 ? '+' : '') + s.anomaly.toFixed(1) + '\u00B0C</td>' +
  '<td style="' + scaleTd + ';color:#888">' + s.hex + '</td>' +
  '<td style="' + scaleTd + ';color:#9ab">' + (s.note || '') + '</td>' +
  '</tr>'
).join('');

// continuous gradient bar showing CIELAB interpolation
const gradStops = [];
for (let i = 0; i <= 40; i++) {
  let t = i / 40;
  let v = -0.5 + t * 2.0;
  gradStops.push(tempScale(v).hex() + ' ' + (t * 100).toFixed(1) + '%');
}
const gradCSS = 'linear-gradient(to right,' + gradStops.join(',') + ')';

'<table style="' + scaleCss + '">' +
  '<thead>' + scaleHeader + '</thead>' +
  '<tbody>' + scaleRows + '</tbody>' +
'</table>' +
'<div style="margin-top:12px">' +
  '<div style="height:14px;border-radius:4px;background:' + gradCSS + '"></div>' +
  '<div style="display:flex;justify-content:space-between;font-family:monospace;font-size:11px;color:#8aa;margin-top:4px">' +
    '<span>\u22120.5\u00B0C</span><span>0.0\u00B0C</span><span>+0.5\u00B0C</span><span>+1.0\u00B0C</span><span>+1.5\u00B0C</span>' +
  '</div>' +
'</div>'

## The Spiral

Each data point maps to a position in polar coordinates: the **month** determines the angle (January at the top, progressing clockwise), and the **temperature anomaly** determines the distance from the center. As the animation progresses year by year, the line traces a spiral that visibly expands outward — especially in the final years, where anomalies routinely exceed +1.0 °C.

Once the spiral completes, a 3D rotation tilts the view 90° to reveal a flat side profile — each year becomes a horizontal band whose width shows how warm it was, with vertical reference lines marking the temperature thresholds. Use the controls at the bottom to pause, scrub through the timeline, or adjust the playback speed.

In [ ]:
let cx = 0, cy = 0, maxR = 1, minR = 0;
let currentIndex = 0;
let playing = true;
let timeline, speedSlider, playBtn;

// 3D cylinder transition
let tiltAngle = 0;
let targetTilt = HALF_PI; // 90° — ends as flat 2D side view
let yearSpacing = 0;
let transitionSteps = Math.round(data.length / 4);

function easeInOutCubic(t) {
  return t < 0.5 ? 4 * t * t * t : 1 - pow(-2 * t + 2, 3) / 2;
}

function polarXY(absMonth, anomaly) {
  let angle = (absMonth / 12) * TWO_PI - HALF_PI;
  let r = map(anomaly, minAnom, maxAnom, minR, maxR);
  let x = cos(angle) * r;
  let y = sin(angle) * r;

  if (tiltAngle > 0.001) {
    let yearIdx = floor(absMonth / 12);
    let z = (yearIdx - (totalYears - 1) / 2) * yearSpacing;
    return {
      x: cx + x,
      y: cy + y * cos(tiltAngle) - z * sin(tiltAngle)
    };
  }

  return { x: cx + x, y: cy + y };
}

function drawReferenceCircles() {
  let fade = tiltAngle > 0.001 ? max(0, 1 - tiltAngle / (targetTilt * 0.3)) : 1;
  if (fade < 0.01) return;

  noFill();
  for (let t of thresholds) {
    let r = map(t, minAnom, maxAnom, minR, maxR);
    let c = tempScale(t).rgb();
    stroke(c[0], c[1], c[2], 25 * fade);
    strokeWeight(0.5);
    circle(cx, cy, r * 2);
    fill(params.labelColor[0], params.labelColor[1], params.labelColor[2], 60 * fade);
    noStroke();
    textSize(11);
    text(t.toFixed(1) + '\u00B0C', cx + r + 6, cy - 4);
  }
}

function drawMonthLabels() {
  // fade out as rings compress toward the side view
  let monthFade = tiltAngle > 0.001 ? max(0, 1 - tiltAngle / (targetTilt * 0.55)) : 1;
  if (monthFade < 0.01) return;

  let labelR = maxR + 28;
  fill(params.labelColor[0], params.labelColor[1], params.labelColor[2], 80 * monthFade);
  noStroke();
  textSize(12);
  textAlign(CENTER, CENTER);

  let blend = constrain(tiltAngle / targetTilt, 0, 1);
  let zBottom = ((totalYears - 1) / 2) * yearSpacing;

  for (let m = 0; m < 12; m++) {
    let angle = (m / 12) * TWO_PI - HALF_PI;
    let lx = cx + cos(angle) * labelR;
    let lyFlat = cy + sin(angle) * labelR;
    let lyTilt = cy + sin(angle) * labelR * cos(tiltAngle) + zBottom * sin(tiltAngle) + 18;
    text(monthNames[m], lx, lerp(lyFlat, lyTilt, blend));
  }
}

## The Cylinder

Once the spiral completes, the view rotates a full 90° around the horizontal axis. The rings separate vertically — oldest at the bottom, newest at the top — passing through a 3D cylinder perspective before settling into a flat 2D side profile. In this final view each year is a horizontal band: wider bands mean warmer years, and vertical reference lines mark the temperature thresholds for easy comparison.

In [ ]:
function drawCylinderScale() {
  if (tiltAngle < 0.1) return;

  let fade = map(tiltAngle, 0.1, targetTilt * 0.5, 0, 1, true);
  if (fade < 0.01) return;

  let halfSpan = ((totalYears - 1) / 2) * yearSpacing;
  let topY = cy - halfSpan * sin(tiltAngle);
  let botY = cy + halfSpan * sin(tiltAngle);
  let pad = 24;

  for (let t of thresholds) {
    let r = map(t, minAnom, maxAnom, minR, maxR);
    let c = tempScale(t).rgb();

    stroke(c[0], c[1], c[2], 45 * fade);
    strokeWeight(0.8);
    line(cx - r, topY - pad, cx - r, botY + pad);
    line(cx + r, topY - pad, cx + r, botY + pad);

    noStroke();
    fill(c[0], c[1], c[2], 150 * fade);
    textSize(10);
    textAlign(CENTER, BOTTOM);
    text(t.toFixed(1) + '\u00B0C', cx - r, topY - pad - 2);
    text(t.toFixed(1) + '\u00B0C', cx + r, topY - pad - 2);
  }

  textAlign(CENTER, CENTER);
}

function drawCylinderLabels() {
  if (tiltAngle < 0.1) return;

  let fade = map(tiltAngle, 0.1, targetTilt * 0.5, 0, 200, true);
  textSize(11);
  noStroke();
  textAlign(LEFT, CENTER);

  let labelInterval = totalYears > 50 ? 20 : totalYears > 20 ? 5 : 1;

  for (let y = 0; y < totalYears; y++) {
    let yr = startYear + y;
    if (yr % labelInterval !== 0 && y !== totalYears - 1) continue;

    let z = (y - (totalYears - 1) / 2) * yearSpacing;
    let yPos = cy - z * sin(tiltAngle);
    let yearStart = y * 12;
    let yearEnd = min(yearStart + 12, data.length);
    let sum = 0;
    for (let i = yearStart; i < yearEnd; i++) sum += data[i].anomaly;
    let avgAnom = sum / (yearEnd - yearStart);
    let c = tempScale(avgAnom).rgb();
    fill(c[0], c[1], c[2], fade);
    text(yr, cx + maxR + 15, yPos);
  }

  textAlign(CENTER, CENTER);
}

## Putting It Together

The sketch below ties everything together into a single animated canvas. It plays in two acts: first the spiral builds month by month, then the view smoothly rotates 90° into a flat side profile. A control bar at the bottom provides play/pause, a timeline scrubber that covers both the spiral and the rotation, and a speed dial.

In [ ]:
function setup() {
  createCanvas(innerWidth, innerHeight);
  textFont('monospace');
  textAlign(CENTER, CENTER);

  cx = width / 2;
  cy = height / 2;
  maxR = min(width, height) * 0.38;
  minR = maxR * 0.18;
  yearSpacing = (min(width, height) * 0.55) / totalYears;

  let totalFrames = data.length + transitionSteps;

  // control bar overlay
  let bar = createDiv('');
  bar.position(0, height - 48);
  bar.style('width', '100%');
  bar.style('display', 'flex');
  bar.style('align-items', 'center');
  bar.style('gap', '10px');
  bar.style('padding', '10px 20px');
  bar.style('background', 'rgba(8, 12, 24, 0.92)');
  bar.style('backdrop-filter', 'blur(8px)');
  bar.style('box-sizing', 'border-box');
  bar.style('font-family', 'monospace');
  bar.style('color', '#b4bed2');
  bar.style('font-size', '13px');
  bar.style('border-top', '1px solid rgba(180,190,210,0.1)');

  // play / pause
  playBtn = createButton('\u23F8');
  playBtn.parent(bar);
  playBtn.style('background', 'none');
  playBtn.style('border', '1px solid rgba(180,190,210,0.3)');
  playBtn.style('color', '#b4bed2');
  playBtn.style('font-size', '16px');
  playBtn.style('cursor', 'pointer');
  playBtn.style('padding', '2px 10px');
  playBtn.style('border-radius', '4px');
  playBtn.mousePressed(() => {
    if (currentIndex >= totalFrames) {
      currentIndex = 0;
      playing = true;
    } else {
      playing = !playing;
    }
    playBtn.html(playing ? '\u23F8' : '\u25B6');
  });

  // timeline scrubber — covers spiral + 3D transition
  timeline = createSlider(0, totalFrames, 0, 1);
  timeline.parent(bar);
  timeline.style('flex', '1');
  timeline.style('accent-color', '#ef8a62');
  timeline.input(() => {
    playing = false;
    playBtn.html('\u25B6');
    currentIndex = timeline.value();
  });

  // speed control
  let speedLabel = createSpan('Speed');
  speedLabel.parent(bar);
  speedSlider = createSlider(0.1, 1.5, 0.8, 0.1);
  speedSlider.parent(bar);
  speedSlider.style('width', '80px');
  speedSlider.style('accent-color', '#67a9cf');
}

function draw() {
  if (width === 0 || !speedSlider) return;

  background(...params.bgColor);

  let totalFrames = data.length + transitionSteps;

  // advance animation — same speed throughout
  if (playing) {
    currentIndex = min(currentIndex + speedSlider.value(), totalFrames);
    timeline.value(floor(currentIndex));

    if (currentIndex >= totalFrames) {
      playing = false;
      playBtn.html('\u25B6');
    }
  }

  // compute tilt from slider position
  if (currentIndex > data.length) {
    let progress = constrain((currentIndex - data.length) / transitionSteps, 0, 1);
    tiltAngle = targetTilt * easeInOutCubic(progress);
  } else {
    tiltAngle = 0;
  }

  drawReferenceCircles();
  drawMonthLabels();

  let limit = floor(min(currentIndex, data.length));

  // draw the spiral
  noFill();
  for (let i = 1; i < limit; i++) {
    let d0 = data[i - 1];
    let d1 = data[i];
    let p0 = polarXY(d0.month + (d0.year - startYear) * 12, d0.anomaly);
    let p1 = polarXY(d1.month + (d1.year - startYear) * 12, d1.anomaly);
    let c = tempScale(d1.anomaly).rgb();

    for (let g = params.glowLayers; g >= 0; g--) {
      let alpha = g === 0 ? 220 : 30 / g;
      let weight = g === 0 ? params.lineWeight : params.lineWeight + g * params.glowSpread;
      stroke(c[0], c[1], c[2], alpha);
      strokeWeight(weight);
      line(p0.x, p0.y, p1.x, p1.y);
    }
  }

  // drawing head (fades during transition)
  if (limit > 0 && limit <= data.length) {
    let headFade = tiltAngle > 0.001 ? max(0, 1 - tiltAngle / (targetTilt * 0.2)) : 1;
    if (headFade > 0.01) {
      let d = data[min(limit - 1, data.length - 1)];
      let p = polarXY(d.month + (d.year - startYear) * 12, d.anomaly);
      let c = tempScale(d.anomaly).rgb();
      noStroke();
      fill(c[0], c[1], c[2], 40 * headFade);
      circle(p.x, p.y, params.dotSize * 4);
      fill(c[0], c[1], c[2], 120 * headFade);
      circle(p.x, p.y, params.dotSize * 2);
      fill(255, 255, 255, 230 * headFade);
      circle(p.x, p.y, params.dotSize * 0.6);
    }
  }

  // cylinder overlay: scale lines + year labels
  drawCylinderScale();
  drawCylinderLabels();

  // center label (fades during transition)
  let centerFade = tiltAngle > 0.001 ? max(0, 1 - tiltAngle / (targetTilt * 0.3)) : 1;
  if (limit > 0 && centerFade > 0.01) {
    let d = data[min(limit - 1, data.length - 1)];
    let c = tempScale(d.anomaly).rgb();
    fill(c[0], c[1], c[2], 200 * centerFade);
    noStroke();
    textSize(28);
    text(d.year, cx, cy - 8);
    fill(params.labelColor[0], params.labelColor[1], params.labelColor[2], 80 * centerFade);
    textSize(13);
    text(monthNames[d.month], cx, cy + 16);
    fill(params.labelColor[0], params.labelColor[1], params.labelColor[2], 40 * centerFade);
    textSize(11);
    text('global temperature anomaly', cx, cy + 34);
  }

  // title in cylinder view
  if (tiltAngle > targetTilt * 0.3) {
    let titleAlpha = map(tiltAngle, targetTilt * 0.3, targetTilt * 0.7, 0, 200, true);
    fill(params.labelColor[0], params.labelColor[1], params.labelColor[2], titleAlpha);
    noStroke();
    textSize(20);
    text('Global Temperature Anomaly', cx, 50);
    textSize(14);
    fill(params.labelColor[0], params.labelColor[1], params.labelColor[2], titleAlpha * 0.6);
    text(startYear + '\u2013' + (startYear + totalYears - 1), cx, 74);
  }

  // attribution
  fill(params.labelColor[0], params.labelColor[1], params.labelColor[2], currentIndex >= data.length ? 70 : 50);
  textSize(14);
  textAlign(LEFT, TOP);
  text('NASA GISS  \u00B7  \u00B0C vs 1951\u20131980 baseline', 20, 20);
  textAlign(CENTER, CENTER);
}

In [ ]:
%show 100% 800px